# Australian Firescar Training Dataset Preparation

This notebook builds a training dataset for fine-tuning OLMo Earth on burned area segmentation.

**Data sources** (via odc-stac from DEA STAC catalog):
- **Labels**: `ga_s2_ba_provisional_3` — Sentinel-2 Burnt Area (delta_nbr used to derive binary mask)
- **Input imagery**: `ga_s2am_ard_3` / `ga_s2bm_ard_3` — Sentinel-2 ARD (6 bands: blue, green, red, NIR, SWIR1, SWIR2)

**Output**: 224×224 chip pairs (imagery + binary burn mask) written to Zarr on S3.

In [ ]:
import os
import numpy as np
import xarray as xr
import odc.stac
from pystac_client import Client
from pathlib import Path

os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'

STAC_URL = 'https://explorer.dea.ga.gov.au/stac'
BA_COLLECTION = 'ga_s2_ba_provisional_3'
S2_COLLECTIONS = ['ga_s2am_ard_3', 'ga_s2bm_ard_3']

# Bands to extract (aligned to OLMo Earth pretrain channels)
INPUT_BANDS = ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir_1', 'nbart_swir_2', 'nbart_swir_3']
LABEL_BAND = 'delta_nbr'

# Chip parameters
CHIP_SIZE = 224
RESOLUTION = 20  # metres (native S2 for SWIR bands)
DNBR_THRESHOLD = 0.27  # threshold for binary burn classification

# Output
OUTPUT_PATH = 's3://easi-dc-data/staging/olmoearth_firescars_chips/'
LOCAL_OUTPUT = Path('/tmp/firescar_chips')  # local fallback if S3 not writable

print('Setup complete')

## 1. Define Study Regions

We sample from fire-prone regions across Australian biomes to ensure diversity.

In [ ]:
# Fire-prone regions spanning different biomes
REGIONS = [
    # (name, bbox [lon_min, lat_min, lon_max, lat_max], time_range)
    ('nt_savanna', [130.0, -14.0, 133.0, -12.0], '2023-06-01/2023-08-31'),
    ('qld_tropical', [143.0, -16.0, 146.0, -14.0], '2023-07-01/2023-10-31'),
    ('nsw_forests', [149.0, -37.0, 151.0, -35.0], '2023-01-01/2023-03-31'),
    ('wa_kimberley', [125.0, -17.0, 128.0, -15.0], '2023-06-01/2023-09-30'),
    ('vic_alpine', [146.0, -38.0, 148.0, -36.5], '2023-01-01/2023-03-31'),
    ('sa_arid', [136.0, -32.0, 139.0, -30.0], '2023-01-01/2023-04-30'),
]

print(f'Defined {len(REGIONS)} study regions')

## 2. Search and Match Burnt Area + S2 ARD Items

For each region, find burnt area items and their corresponding S2 ARD scenes (same tile, same date).

In [ ]:
catalog = Client.open(STAC_URL)

def find_paired_items(bbox, datetime_range, max_ba_items=50):
    """Find burnt area items and matching S2 ARD for the same tile/date."""
    ba_results = catalog.search(
        collections=[BA_COLLECTION],
        bbox=bbox,
        datetime=datetime_range,
        max_items=max_ba_items,
    )
    ba_items = list(ba_results.items())
    
    pairs = []
    for ba_item in ba_items:
        # Extract tile ID and date from the BA item
        dt = ba_item.datetime
        date_str = dt.strftime('%Y-%m-%d')
        
        # Search for matching S2 ARD on the same date and bbox
        item_bbox = list(ba_item.bbox)
        s2_results = catalog.search(
            collections=S2_COLLECTIONS,
            bbox=item_bbox,
            datetime=f'{date_str}/{date_str}',
            max_items=2,
        )
        s2_items = list(s2_results.items())
        if s2_items:
            pairs.append((ba_item, s2_items[0]))
    
    return pairs

# Collect all pairs across regions
all_pairs = []
for name, bbox, dt_range in REGIONS:
    pairs = find_paired_items(bbox, dt_range)
    print(f'{name}: {len(pairs)} paired scenes')
    all_pairs.extend([(name, ba, s2) for ba, s2 in pairs])

print(f'\nTotal paired scenes: {len(all_pairs)}')

## 3. Load and Chip a Scene Pair

For each paired scene, load both datasets at matching resolution, derive the binary burn mask, and tile into 224×224 chips.

In [ ]:
def load_scene_pair(ba_item, s2_item, resolution=RESOLUTION):
    """Load burnt area label and S2 imagery for a matched pair."""
    # Load burnt area delta_nbr at target resolution
    ba_ds = odc.stac.load(
        [ba_item],
        bands=[LABEL_BAND],
        resolution=resolution,
        chunks={'x': 2048, 'y': 2048},
    )
    
    # Load S2 ARD bands aligned to same geobox (like= sets resolution and CRS)
    s2_ds = odc.stac.load(
        [s2_item],
        bands=INPUT_BANDS,
        chunks={'x': 2048, 'y': 2048},
        like=ba_ds.odc.geobox,
    )
    
    return s2_ds, ba_ds


def make_burn_mask(ba_ds, threshold=DNBR_THRESHOLD):
    """Convert delta_nbr to binary burn mask (1=burned, 0=unburned)."""
    dnbr = ba_ds[LABEL_BAND].isel(time=0)
    mask = (dnbr > threshold).astype(np.uint8)
    # Mark nodata (NaN in source) as 255
    mask = mask.where(~np.isnan(dnbr), 255)
    return mask


def chip_scene(imagery, mask, chip_size=CHIP_SIZE, min_burn_frac=0.01, max_nodata_frac=0.1):
    """Tile a scene into chips, filtering by burn fraction and nodata."""
    h, w = mask.shape
    chips = []
    
    for y0 in range(0, h - chip_size + 1, chip_size):
        for x0 in range(0, w - chip_size + 1, chip_size):
            mask_chip = mask[y0:y0+chip_size, x0:x0+chip_size].values
            
            # Skip chips with too much nodata
            nodata_frac = (mask_chip == 255).mean()
            if nodata_frac > max_nodata_frac:
                continue
            
            # Check burn fraction (only among valid pixels)
            valid = mask_chip[mask_chip != 255]
            if len(valid) == 0:
                continue
            burn_frac = valid.mean()
            if burn_frac < min_burn_frac:
                continue
            
            # Extract imagery chip (all bands)
            img_chip = imagery.isel(time=0)[:, y0:y0+chip_size, x0:x0+chip_size].values  # (bands, h, w)
            
            # Skip if imagery has nodata
            if np.any(img_chip == -999):
                img_nodata_frac = (img_chip[0] == -999).mean()
                if img_nodata_frac > max_nodata_frac:
                    continue
            
            # Set mask nodata pixels to 0 (unburned) for training
            mask_clean = mask_chip.copy()
            mask_clean[mask_clean == 255] = 0
            
            chips.append({
                'imagery': img_chip.astype(np.int16),
                'mask': mask_clean.astype(np.uint8),
                'burn_fraction': float(burn_frac),
            })
    
    return chips

print('Chip functions defined')

## 4. Process All Scenes

Iterate over paired scenes, generate chips, and collect into arrays.

In [ ]:
all_imagery = []
all_masks = []
all_metadata = []

for i, (region_name, ba_item, s2_item) in enumerate(all_pairs):
    print(f'[{i+1}/{len(all_pairs)}] Processing {region_name} - {ba_item.datetime.strftime("%Y-%m-%d")}...')
    
    try:
        s2_ds, ba_ds = load_scene_pair(ba_item, s2_item)
        
        # Compute (load from dask)
        s2_ds = s2_ds.compute()
        ba_ds = ba_ds.compute()
        
        # Create binary mask
        mask = make_burn_mask(ba_ds)
        
        # Stack imagery bands into (bands, y, x)
        imagery = xr.concat([s2_ds[b] for b in INPUT_BANDS], dim='band')
        
        # Generate chips
        chips = chip_scene(imagery, mask)
        print(f'  -> {len(chips)} chips (burn fracs: {[f"{c["burn_fraction"]:.2f}" for c in chips[:5]]}...)')
        
        for chip in chips:
            all_imagery.append(chip['imagery'])
            all_masks.append(chip['mask'])
            all_metadata.append({
                'region': region_name,
                'date': ba_item.datetime.isoformat(),
                'burn_fraction': chip['burn_fraction'],
            })
    except Exception as e:
        print(f'  -> FAILED: {e}')
        continue

print(f'\nTotal chips collected: {len(all_imagery)}')

## 5. Normalise and Split

In [ ]:
import json

# Stack into arrays
imagery_arr = np.stack(all_imagery)  # (N, 6, 224, 224)
masks_arr = np.stack(all_masks)      # (N, 224, 224)

print(f'Imagery shape: {imagery_arr.shape}')
print(f'Masks shape: {masks_arr.shape}')
print(f'Burn pixel fraction: {masks_arr.mean():.4f}')

# Compute per-band statistics (excluding nodata=-999)
band_stats = {}
for i, band_name in enumerate(INPUT_BANDS):
    valid = imagery_arr[:, i][imagery_arr[:, i] != -999].astype(np.float32)
    band_stats[band_name] = {
        'mean': float(valid.mean()),
        'std': float(valid.std()),
        'min': float(valid.min()),
        'max': float(valid.max()),
    }
    print(f'  {band_name}: mean={band_stats[band_name]["mean"]:.1f}, std={band_stats[band_name]["std"]:.1f}')

# Stratified split by region (70/15/15)
from sklearn.model_selection import train_test_split

regions = [m['region'] for m in all_metadata]
indices = np.arange(len(all_metadata))

train_idx, temp_idx = train_test_split(indices, test_size=0.3, stratify=regions, random_state=42)
temp_regions = [regions[i] for i in temp_idx]
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=temp_regions, random_state=42)

print(f'\nSplit: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}')

## 6. Write to Zarr

In [ ]:
import zarr

# Write locally (switch to S3 path when credentials allow)
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
output_path = str(LOCAL_OUTPUT)

store = zarr.open(output_path, mode='w')

for split_name, split_idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    grp = store.create_group(split_name)
    grp.create_dataset('imagery', data=imagery_arr[split_idx], chunks=(16, 6, 224, 224), dtype='int16')
    grp.create_dataset('masks', data=masks_arr[split_idx], chunks=(16, 224, 224), dtype='uint8')
    print(f'{split_name}: {len(split_idx)} chips written')

# Save metadata
metadata = {
    'band_names': INPUT_BANDS,
    'band_stats': band_stats,
    'chip_size': CHIP_SIZE,
    'resolution_m': RESOLUTION,
    'dnbr_threshold': DNBR_THRESHOLD,
    'num_chips': {'train': len(train_idx), 'val': len(val_idx), 'test': len(test_idx)},
    'regions': list(set(regions)),
    'label_source': BA_COLLECTION,
    'imagery_source': S2_COLLECTIONS,
}

with open(LOCAL_OUTPUT / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\nDataset written to: {output_path}')
print(f'Metadata: {LOCAL_OUTPUT / "metadata.json"}')

## 7. Quick Visualisation

In [ ]:
import matplotlib.pyplot as plt

# Show a few sample chips
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
sample_idx = np.random.choice(train_idx, 3, replace=False)

for row, idx in enumerate(sample_idx):
    img = imagery_arr[idx]
    msk = masks_arr[idx]
    
    # RGB (bands 2,1,0 = red, green, blue)
    rgb = np.clip(img[[2,1,0]].transpose(1,2,0) / 3000.0, 0, 1)
    # False colour (SWIR1, NIR, Red)
    fc = np.clip(img[[4,3,2]].transpose(1,2,0) / 3000.0, 0, 1)
    # NBR-like (NIR, SWIR2)
    nir = img[3].astype(np.float32)
    swir2 = img[5].astype(np.float32)
    nbr = np.where((nir + swir2) != 0, (nir - swir2) / (nir + swir2), 0)
    
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f'RGB (chip {idx})')
    axes[row, 1].imshow(fc)
    axes[row, 1].set_title('False Colour (SWIR/NIR/R)')
    axes[row, 2].imshow(nbr, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
    axes[row, 2].set_title('NBR')
    axes[row, 3].imshow(msk, cmap='Reds', vmin=0, vmax=1)
    axes[row, 3].set_title(f'Burn Mask ({msk.mean():.2%} burned)')

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Upload to S3 (Production)

When running with appropriate IAM credentials (e.g., from a Dask worker or Argo pod), write directly to S3:

In [ ]:
# Uncomment when running with S3 write access:
#
# import s3fs
# fs = s3fs.S3FileSystem()
# s3_store = zarr.open(s3fs.S3Map(OUTPUT_PATH, s3=fs), mode='w')
#
# for split_name, split_idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
#     grp = s3_store.create_group(split_name)
#     grp.create_dataset('imagery', data=imagery_arr[split_idx], chunks=(16, 6, 224, 224), dtype='int16')
#     grp.create_dataset('masks', data=masks_arr[split_idx], chunks=(16, 224, 224), dtype='uint8')
#
# # Upload metadata
# with fs.open(OUTPUT_PATH + 'metadata.json', 'w') as f:
#     json.dump(metadata, f, indent=2)
#
# print(f'Uploaded to {OUTPUT_PATH}')